# VLA Basics

## 0. The Core Task of VLA

VLA stands for **Vision-Language-Action model**. Its core task is:

**Given visual observations, the robot's own state, and a natural-language instruction, output robot actions that can complete the task.**

The simplest form is:

```text id="2zpa54"
Input:
    image / video observation
    robot state / proprioception
    language instruction

Output:
    robot action
```

It can be written as:

$$
a_t = \pi_\theta(I_t, L, s_t)
$$

If the model outputs a future action sequence, then:

$$
A_t = (a_t, a_{t+1}, \ldots, a_{t+H})
$$

$$
A_t \sim p_\theta(A_t \mid I_t, L, s_t)
$$

where:

```text id="5i1wzz"
    I_t = current visual observation
    L = language instruction
    s_t = robot state
    A_t = a future action chunk
```

At a deeper level, VLA is not merely a mapping from “image + text → action”. It tries to learn an **affordance-level multimodal action-planning capability**:

```text id="vqeuus"
    perceive objects
    understand the language goal
    infer the object's manipulability
    select the object and interaction region
    generate a short-horizon action trajectory
    execute it through a low-level controller
```

For example:

```text id="mqtjgt"
Instruction: put the red cup into the blue bowl

The VLA needs to understand:
1. the red cup is the object to manipulate
2. the blue bowl is the target container
3. the cup can be grasped
4. the bowl can serve as a place-into target
5. the current gripper position determines the next action
6. the output should include actions such as approaching, grasping, carrying, and releasing
```

Therefore, the essence of VLA is neither pure visual recognition nor pure language understanding, but rather:

**a policy model that connects visual semantics, language semantics, robot embodiment state, and action affordances.**

---

# 1. Basic Structure of VLA

A basic VLA can be written as:

```text id="sxdq07"
    observation
        ↓
    modality-specific encoder
        ↓
    continuous embeddings
        ↓
    LLM / Transformer / multimodal backbone
        ↓
    hidden states / condition representation
        ↓
    action head
        ↓
    action
```

More concretely:

```text id="ejzz3m"
    image
    ↓ vision encoder
    visual embeddings

    language instruction
    ↓ tokenizer + embedding layer
    text embeddings

    robot state
    ↓ MLP / state encoder
    state embeddings

    visual embeddings + text embeddings + state embeddings
    ↓ multimodal Transformer / VLA backbone
    condition representation / hidden states

    ↓ action head
    action vector / action token / action chunk
```

One terminology issue is especially important:

**Strictly speaking, a token refers to a discrete symbol or a discrete index, whereas an embedding is a continuous vector.**

In text:

```text id="a8s57r"
"cup" → token id → embedding vector
```

In images:

```text id="ev1slg"
image patch → vision encoder → visual feature → projector → visual embedding
```

Therefore, papers often say “visual tokens” or “state tokens”, but more rigorously, they are:

```text id="dapn17"
token-like continuous embeddings
```

or:

```text id="656got"
continuous modality representations occupying positions in a Transformer sequence
```

---

# 2. How Do Input Data Enter an LLM / Transformer?

## 2.1 Image Input

Images cannot be fed directly into an LLM. A common pipeline is:

```text id="wm0z03"
image
  ↓
patchify / vision encoder
  ↓
patch-level visual features
  ↓
projection layer
  ↓
visual embeddings
```

For example, a ViT splits an image into patches and produces a vector for each patch. A projector then maps these vectors to the same dimensionality as the LLM hidden size.

```text id="cfn0cu"
vision feature dim: 1024
LLM hidden dim: 4096

visual feature
  ↓ Linear projector
visual embedding with dim 4096
```

These visual embeddings can be placed in the same Transformer sequence as text embeddings.

---

## 2.2 Language Input

A language instruction is converted into token ids by a tokenizer, and then into continuous vectors through an embedding table:

```text id="31m1mg"
"pick up the red cup"
        ↓ tokenizer
token ids
        ↓ embedding layer
text embeddings
```

Language provides the task condition:

```text id="4klxfn"
Which object should be picked?
Should the robot push, grasp, place, open, or wipe?
What are the target object and target location?
```

Without language, the robot only sees the world. With language, the robot knows the current task goal.

---

## 2.3 Robot State Input

The robot state usually includes:

```text id="ctuukz"
    joint angles q
    joint velocities dq
    end-effector position
    end-effector orientation
    gripper open/closed state
    force / tactile feedback
    action history
```

These are continuous values and are usually encoded by an MLP:

```text id="t4nkk6"
    robot state
    ↓ normalization
    ↓ MLP / linear projector
    state embedding
```

If a history of states is used, for example:

$$
s_{t-2},\ s_{t-1},\ s_t
$$

then we can obtain a state embedding sequence:

$$
e^s_{t-2},\ e^s_{t-1},\ e^s_t
$$

The role of state is to tell the model:

```text id="4qrkes"
Where is the robot itself?
Is the gripper open or closed?
Has it already approached the target?
Which stage of the task is it currently in?
```

Images alone are not enough, because under the same image, different robot states may require completely different next actions.

---

# 3. How Are Multimodal Embeddings Organized?

The simplest method is concatenation:

```text id="v4zf03"
    visual embeddings + state embeddings + text embeddings
            ↓
    Transformer self-attention
```

Special tokens can also be used to explicitly mark modality boundaries:

```text id="lqoxcu"
    <image>
    visual embeddings
    </image>

    <state>
    state embeddings
    </state>

    <instruction>
    text embeddings
    </instruction>
```

For multi-camera input, view embeddings can also be added:

```text id="4d3r9v"
    <front_camera>
    front visual embeddings
    </front_camera>

    <wrist_camera>
    wrist visual embeddings
    </wrist_camera>
```

For historical frames, time embeddings can be added:

$$
I_{t-2},\ I_{t-1},\ I_t
$$

For different robots, embodiment embeddings can be added:

```text id="rz65pd"
    robot_type = Franka / UR5 / ALOHA / mobile manipulator
```

Therefore, an LLM / Transformer usually does not just receive an “unlabeled mixed sequence”. Instead, it distinguishes modalities through:

```text id="aw2fjv"
    special tokens
    modality embeddings
    position embeddings
    time embeddings
    camera-view embeddings
    robot embodiment embeddings
```

---

# 4. Is the Output a Single Modality Channel or Multiple Channels?

This depends on the architecture.

## 4.1 Single-Channel Unified Sequence

Some VLA models discretize actions into action tokens, so text tokens and action tokens are generated within one unified sequence.

The structure is similar to:

```text id="skzf2j"
    image + instruction
            ↓
           LLM
            ↓
    action tokens
```

The advantage is that the autoregressive LLM framework can be reused.

The disadvantage is that actions are originally continuous quantities. Discretizing them into tokens can cause precision loss and slower generation.

---

## 4.2 Multi-Head Output

Another option is to share the backbone while assigning different output heads to different tasks:

```text id="r8q7gc"
shared VLA backbone
    ├── language head
    ├── action head
    ├── subtask prediction head
    ├── object detection / grounding head
```

This type of structure is more suitable for multi-task co-training.

---

## 4.3 Flow / Diffusion Action Head

In modern continuous-control VLA models, the action head may not directly output the final action. Instead, it may output the correction direction during a generative process.

The structure is:

```text id="t9fn75"
    image + language + state
            ↓
    VLA / VLM backbone
            ↓
    condition representation c

    noisy action chunk + flow time τ + c
            ↓
    flow action head
            ↓
    velocity / denoising direction

    after multi-step updates:
    action chunk
```

---

# 5. Semantic Alignment: How Are Different Modalities Aligned to the “World at a Given Moment”?

The core difficulty of VLA is not simply concatenating different embeddings, but making them semantically and behaviorally aligned.

## 5.1 Vision-Language Alignment

Many VLA models do not learn vision-language alignment from scratch. Instead, they inherit pretrained capabilities from VLMs.

A VLM has already learned from large-scale image-text tasks that:

```text id="6jwwxb"
the red cup in the image ↔ the text "red cup"
the drawer in the image ↔ the text "drawer"
the bowl in the image ↔ the text "bowl"
```

Common training tasks include:

```text id="ixkswu"
    image captioning
    VQA
    image-text matching
    contrastive learning
    object detection
    referring expression grounding
```

Therefore, VLA usually starts from a VLM and then adds robot action data, turning the model from “seeing and describing” into “seeing and acting”.

---

## 5.2 Alignment Through the Projection Layer

The output space of an image encoder is different from the hidden space of an LLM, so a projector is needed:

```text id="mwyf3q"
    visual feature
        ↓
    projection layer
        ↓
    LLM-compatible visual embedding
```

LLaVA-style models are a typical example: the vision encoder provides image features, the projection layer maps them into the embedding space that the language model can process, and then the projected visual embeddings enter the LLM together with text embeddings.

However, one point must be emphasized:

**Projecting to the same hidden dimension does not automatically mean semantic alignment.**

Real alignment comes from training objectives and data distributions.

---

## 5.3 Alignment Through Self-Attention / Cross-Attention

Concatenation-based input relies on self-attention:

```text id="q2ecot"
    visual embeddings + text embeddings + state embeddings
            ↓
    self-attention
```

In this way, the word `"cup"` can attend to the cup region in the image, and the state embedding can be aligned with the visual embedding of the current gripper state.

Another method is cross-attention:

```text id="hm9m7t"
    text/state/action query
            attends to
    visual key/value
```

This is equivalent to letting language or action representations actively read visual information.

---

## 5.4 Alignment Through Behavioral Supervision

A VLM can only tell the model:

```text id="0z9xnt"
    What object is this?
    Roughly where is it?
```

But a VLA must also learn:

```text id="dicfse"
    How should the robot approach it?
    How should it grasp it?
    When should the gripper close?
    What will happen after the action?
```

This part comes from robot demonstrations.

A training sample is usually:

```text id="9ou7ss"
    image_t
    instruction
    robot_state_t
    expert_action_t or expert_action_chunk_t
```

Through imitation learning, the model learns:

```text id="g0tz58"
language goal ↔ visual object ↔ robot state ↔ action outcome
```

For example:

```text id="y2qtsn"
    "pick up the red cup"
    + the red cup is in the front-left area
    + the gripper is on the right side and open
    → move toward the front-left, approach the cup, close the gripper, and lift it
```

This is the deeper grounding in VLA:

```text id="juiak1"
word ↔ object ↔ affordance ↔ action consequence
```

---

## 5.5 Auxiliary Losses

To prevent the model from only learning superficial correlations, auxiliary losses can be added:

```text id="ygyxrm"
    object detection loss
    bounding box grounding loss
    subtask prediction loss
    VQA / captioning loss
    image-text matching loss
    future state prediction loss
    action prediction loss
```

These auxiliary tasks help the model explicitly represent:

```text id="h8s0wq"
    Where is the target object?
    Which object does the language refer to?
    What is the current task stage?
    Which subtask should be performed next?
    ...
```

---

# 6. What Is Affordance?

Affordance can be understood as:

```text id="e1pp7s"
    manipulability
    action possibility
    interaction possibility
```

It refers to:

**the possible ways an object or environment allows a particular agent to interact with it.**

For example:

```text id="rvw8if"
Chair:
    seat → sit
    backrest → lean against
    armrest → support the hand

Cup:
    body / handle → grasp
    mouth / inner space → contain liquid or objects
    whole cup → move / pour water

Drawer:
    handle → grasp / pull
    drawer space → put in / take out

Button:
    surface → press
```

Therefore, affordance is not simply the object category. Rather, it asks:

```text id="etlf2n"
    What can I do with this object in the current state?
    Which part is interactable?
    What will happen after taking an action?
```

For VLA, affordance is the key intermediate layer that connects language and action.

For example:

```text id="p8eia7"
    "open the drawer"
```

The model cannot merely recognize the drawer. It must also know that:

```text id="izxq44"
    the drawer handle is a graspable / pullable part
    the drawer has a sliding direction
    opening requires approaching the handle, grasping it, and pulling outward
```

Therefore, the goal of VLA is not simply to learn:

```text id="3sie5p"
    object label → action
```

but to learn:

```text id="2b2eb9"
    language instruction
        ↓
    visual object grounding
        ↓
    affordance recognition
        ↓
    local action planning
        ↓
    continuous control
```

---

# 7. Is VLA Autoregressive?

The answer is:

**It can be autoregressive, but it does not have to be.**

The key depends on how actions are represented.

---

## 7.1 The Autoregressive Action-Token Route

If continuous actions are discretized into action tokens, the model can generate them autoregressively like a language model:

```text id="xk41ry"
image + instruction + state
    → action_token_1
    → action_token_2
    → action_token_3
    → ...
```

The form is similar to:

$$
p(z_1, z_2, \ldots, z_N \mid c)
=
\prod_i p(z_i \mid z_{<i}, c)
$$

where:

```text id="gxmdzm"
z_i = action token
c = conditioning variable, such as image, language, and state
```

Advantages:

```text id="xnrfsi"
    reuses the LLM framework
    uses a simple training objective
    can place actions and language into a unified token sequence
```

Disadvantages:

```text id="v85s36"
    continuous actions are discretized
    action precision is affected by binning
    the previous token must be generated before the next one
    inference speed is limited
    high-frequency control becomes unnatural
```

This generation method has temporal constraints similar to an RNN. Although it uses a Transformer, the generation process is still sequential decoding, so it cannot obtain a complete action sequence in parallel in one step.

---

## 7.2 The Non-Autoregressive Action-Chunk Route

Another class of methods directly generates a future action chunk:

$$
A_t = (a_t, a_{t+1}, \ldots, a_{t+H})
$$

This is called action chunking.

Advantages:

```text id="5x8r9y"
actions are smoother
short-term motion intent can be represented
jitter in one-step behavior cloning can be reduced
it is suitable for continuous control
it can reduce the pressure of high-frequency inference on large models
```

Disadvantages:

```text id="dw09q7"
actions farther into the future are less reliable
long horizons are prone to drift
after contact occurs, later actions may become invalid
in real execution, usually only the first few steps are used
```

Therefore, an action chunk is not a “complete open-loop trajectory”, but rather:

```text id="sfie8y"
a short-horizon local trajectory + intent representation
```

---

# 8. How Do Flow Matching / Diffusion Policy Generate an Action Chunk?

## 8.1 Direct Action Head

A standard VLA can directly output an action chunk:

```text id="ppx0mo"
    condition c
        ↓
    action head
        ↓
    A_t
```

Formula:

$$
A_t = f_\theta(c)
$$

where:

```text id="epijwb"
c = condition representation encoded by the VLA backbone
```

During training, MSE is used:

$$
\mathcal{L} = \left| A_{\mathrm{pred}} - A_{\mathrm{expert}} \right|^2
$$

This method is simple, but it can easily learn averaged actions.

---

## 8.2 Flow Matching Action Head

Flow matching does not directly output the final action. Instead, it learns:

```text id="73xf7g"
how to transform a noisy action chunk into a reasonable action chunk
```

The structure is:

```text id="bw5fpl"
    image + language + state
            ↓
    VLA backbone
            ↓
    condition c

    noisy action chunk X_τ
    + flow time τ
    + condition c
            ↓
    flow action head
            ↓
    velocity vθ

    multi-step integration
            ↓
    action chunk A
```

The formula is:

$$
\frac{dX_\tau}{d\tau}
=
v_\theta(X_\tau, \tau, c)
$$

where:

```text id="r6ar3h"
X_τ = the noisy action chunk at the current denoising step
τ = flow / denoising time
c = image-language-state condition
vθ = correction direction / velocity field
```

Finally:

```text id="e1j7ke"
X_0 = random noise action chunk
X_1 = generated action chunk
```

Therefore, in a flow-matching-based VLA:

```text id="3hm0ai"
    the VLA backbone mainly outputs a condition representation
    the flow action head generates continuous actions under this condition
```

The overall model still outputs an action chunk, but internally it is not:

```text id="no92da"
    condition → action
```

but rather:

```text id="gvfom0"
    condition + noise → action distribution
```

---

## 8.3 What Exactly Is the Noise?

The noise is not sensor noise and not robot jitter.

In action diffusion / flow for VLA, noise is:

```text id="wewbnf"
a random array with the same shape as the action chunk
```

If the real action is:

$$
A \in \mathbb{R}^{H \times d_a}
$$

then the noise is also:

$$
Z \in \mathbb{R}^{H \times d_a}
$$

where:

```text id="3c5rur"
H = action horizon
d_a = action dimension
```

The difference is:

```text id="na17k0"
real actions come from the expert demonstration distribution
noise comes from a standard Gaussian distribution
```

During training, the model learns how to transform the noise distribution into the real action distribution.

---

## 8.4 Why Start from Noise Instead of a Rough Trajectory?

Starting from noise has several advantages:

```text id="7ef91k"
the starting point is simple
it does not depend on an external planner
it can represent multimodal distributions
different noises can generate multiple candidate trajectories
```

For example:

```text id="voias0"
z1 → grasp from the left side
z2 → grasp from the right side
z3 → grasp from above
```

If the model starts from a rough trajectory, then it becomes more like:

```text id="nn31nz"
trajectory refinement
guided diffusion
residual correction
```

This is certainly possible and very reasonable in engineering. However, it requires another system to first provide the rough trajectory, such as a motion planner, a heuristic method, or the previous action chunk.

Therefore:

```text id="mhbq8t"
pure-noise starting point:
    suitable for fully modeling p(action_chunk | condition)

rough-trajectory starting point:
    suitable for engineering warm start / refinement
```

---

# 9. How Many Steps in a Long Action Sequence Can Actually Be Used?

During training, the model may output:

$$
H = 16,\ 32,\ 64
$$

But during real deployment, the entire action chunk is usually not executed open-loop. Instead:

```text id="lmhadg"
predict the next H steps
execute only the first k steps
observe again
predict again
```

This is called receding horizon control.

For example:

```text id="6x7ca4"
t = 0:
    predict a_0 to a_15
    execute a_0 to a_3

t = 4:
    observe again
    predict a_4 to a_19
    execute a_4 to a_7
```

Usually:

```text id="3ka2q9"
    first few steps: executable actions
    middle steps: short-term trajectory reference
    later steps: intent / trend
```

The farther into the future an action is, the more easily it is affected by execution error, contact changes, occlusion, object sliding, and other factors.

Therefore, the role of action chunking is not to let the robot execute the entire chunk open-loop, but to:

```text id="w882wn"
    allow the model to learn short-term planning and trajectory smoothness
    while maintaining closed-loop error correction through frequent replanning
```

---

# 10. How Accurate Is VLA Control?

This needs to be understood layer by layer.

If VLA is treated as:

```text id="7j4i4b"
    a millimeter-level low-level controller
```

then it is currently difficult for it to outperform traditional control, motion planning, visual servoing, force control, impedance control, or specialized RL policies.

But if VLA is treated as:

```text id="jo9xuv"
    an affordance-level high-dimensional semantic-action planner
```

then it is valuable.

VLA is good at:

```text id="0hrb4o"
    selecting target objects according to language
    understanding task semantics
    deciding which object to manipulate in multi-object scenes
    learning common manipulation patterns from demonstrations
    generating coarse-to-medium-precision local actions
```

VLA is not good at:

```text id="tpe2ew"
    millimeter-level insertion
    high-speed dynamic control
    strong-contact force control
    strict stability requirements
    high-precision industrial assembly
```

Major sources of error include:

```text id="2xjown"
    visual localization error
    language grounding error
    action representation error
    coordinate-frame inconsistency
    action discretization error
    horizon drift
    insufficient contact feedback
    training-data distribution bias
    LLM / VLM semantic hallucination or reasoning error
```

LLMs can make mistakes in basic arithmetic, which shows that they are not strict symbolic calculators. Similarly, VLAs are not strict controllers. Their strength is not “precisely computing every control value”, but rather:

```text id="0zw8bm"
    generating a roughly reasonable action distribution from multimodal context
```

A truly reliable system usually needs:

```text id="9rgfop"
    VLA / VLM: understand the task and affordances
    motion planner: ensure geometric feasibility and collision avoidance
    low-level controller: ensure stable execution, speed limits, and force control
    visual servo / tactile feedback: perform fine-grained closed-loop correction
```

---

# 11. Who Is Responsible for Action Planning?

“Planning” in VLA can be divided into multiple layers:

```text id="3azf3k"
    high-level task planning:
        what to do, and in what order

    mid-level affordance / skill planning:
        which object to manipulate, which part to interact with, and what the next-stage goal is

    low-level local trajectory generation:
        how dx dy dz / rotation / gripper should move over the next H steps

    low-level control execution:
        IK, joint control, force control, impedance control, and safety constraints
```

Flow matching or diffusion policy is mainly responsible for:

```text id="2xaetp"
    low-level local action chunk generation
```

The VLA backbone is responsible for:

```text id="6e9i7e"
    understanding the current scene, language instruction, and robot state
    providing the condition representation
```

A high-level LLM / VLM planner or task policy is responsible for:

```text id="owax8x"
    long-horizon task decomposition
    selecting the next subtask
    replanning after failure
```

A traditional motion planner / controller is responsible for:

```text id="bkjh00"
    reachability
    collision avoidance
    velocity / acceleration constraints
    force control
    safe execution
```

Therefore, the most reliable interpretation is not:

```text id="qc6vpn"
    VLA = all planning + all control
```

but rather:

```text id="erf2oh"
    VLA = multimodal semantic understanding + affordance inference + local action policy
```

---

# 12. How Is Generality Guaranteed?

The generalization ability of VLA mainly comes from several sources.

## 12.1 VLM Pretraining

A VLM inherits internet-scale image-text knowledge and can recognize many objects, scenes, and language expressions.

This means VLA does not need to learn all semantics from robot data alone from scratch.

---

## 12.2 Large-Scale Robot Demonstration Data

Robot data provides information about:

```text id="xy5mkq"
    how actions are executed
    how objects are contacted
    when the gripper should close
    how task stages progress
```

This part cannot be obtained from image-text pretraining alone; it must come from real or simulated robot manipulation data.

---

## 12.3 Co-Training on Heterogeneous Data

VLA can improve generalization through multi-source data, such as:

```text id="eh4rzs"
    multi-robot data
    multi-task demonstrations
    web image-text data
    object detection data
    subtask prediction data
    language instruction data
    failure recovery data
```

This shows that generalization does not come from model architecture alone, but from:

```text id="l4pkyv"
    data diversity
    task diversity
    modality-supervision diversity
    environment diversity
```

---

## 12.4 Avoiding Spurious Correlations

If the training set always contains:

```text id="5zc2b2"
    the red cup is on the left
    the blue bowl is on the right
```

the model may learn:

```text id="k1l98q"
    red cup = the object on the left
```

instead of truly grounding to the red cup.

Therefore, it needs:

```text id="zs1q1f"
    randomized object positions
    disentangled color / shape / position
    diverse language expressions
    counterfactual samples
    occlusion and lighting variations
    multi-view input
    failure recovery data
```

This is very close to the idea of ROP:

**If the surface order is fixed, the agent may learn shortcuts; only by randomizing observation order or disentangling attributes is it more likely to learn true conceptual correspondences.**

---

# 13. Final Summary

VLA can be understood as:

**an embodied policy model conditioned on vision, language, and robot state.**

Its core is not to let an LLM directly and precisely compute motor control values, but to let the model learn:

```text id="lmk6hb"
    language goal
        ↓
    visual grounding
        ↓
    object affordance
        ↓
    task stage
        ↓
    local action trajectory
```

Therefore, VLA is best viewed as:

```text id="ggra68"
    affordance-level semantic action planner
    +
    local visuomotor policy
```

rather than:

```text id="s0xrbr"
    a universal low-level controller that completely replaces traditional control algorithms
```

In a flow matching / diffusion policy architecture, the VLA backbone is more like a condition encoder:

```text id="lvu98g"
    image + language + state
            ↓
    condition representation
```

The action generator then uses this condition to gradually transform a noisy action chunk into a reasonable continuous action trajectory:

```text id="q9z4ej"
    noise action chunk
    + condition
            ↓
    flow / diffusion action head
            ↓
    action chunk
```

During real execution, usually only the first few steps of the action chunk are executed. The robot then observes again and regenerates actions to maintain closed-loop control.

Therefore, the proper positioning of VLA is:

```text id="8o0u7m"
    high-level semantic understanding
    + affordance inference
    + short-horizon action generation
    + closed-loop execution by a low-level controller
```

This is why VLA matters in robotics:
not because it can precisely replace traditional control, but because it provides a way to unify **language, vision, world semantics, object manipulability, and robot action** within a single learning framework.

---
